# 04 Agent Graph

**Owner.** Tim (Agent Orchestration & Safety Engineer) · **Supports.** Jack

**Goal.** The full flow as a LangGraph graph: router, two specialist agents, reviewer loop and a hand-over to a
human. The process becomes visible, controllable and testable.

**Definition of Done.** One employment and one tenancy case run end to end. The stop criteria of the reviewer loop
are documented (tasks T10 and T14).

## 1. Why a graph?

A linear chain runs every step exactly once. We need a **loop** (the reviewer sends a draft back), **branches**
(employment or tenancy law, or out of scope) and a **stop rule** (at most two revision rounds, then a human).
LangGraph models this explicitly: a shared state, nodes that change it and edges that decide what comes next.

## 2. Setup

In [1]:
import os, json, re, pathlib
from typing import TypedDict
from dotenv import load_dotenv

load_dotenv()
LLM_BASE_URL = "https://litellm.engines.aisl.science/v1"
LLM_MODEL = "azure_ai/gpt-oss-120b"
MAX_REVISIONS = 2

## 3. The state

Everything the nodes need to share lives in one typed dictionary. Each node returns only the fields it changes.

In [3]:
class LegalState(TypedDict, total=False):
    question: str        # the case as typed by the user
    area: str            # arbeitsrecht | mietrecht | out_of_scope
    sources: list        # retrieved articles: id, title, text
    draft: str           # current answer draft
    review: str          # last reviewer verdict: "OK" or "REVISE: <reason>"
    revisions: int       # number of rounds the reviewer sent back
    needs_human: bool    # True if the case goes to a human
    answer: str          # final answer shown to the user

## 4. Retrieval for now

Until notebook 02 and the MCP server are wired in (W4), the graph uses a small keyword search over
`data/or_articles.jsonl`. The function signature stays the same, so swapping it later changes nothing else.

In [4]:
from rank_bm25 import BM25Okapi

def _tokenize(text):
    return re.findall(r"\w+", text.lower())

# T11: the knowledge base comes from the repository (GitHub integration swiss_legal_agent), not a local copy.
import sys
sys.path.insert(0, str(pathlib.Path("swiss_legal_agent").resolve()))
from lexgraph.rag.ingest import ARTICLES_FILE, load_articles

def make_keyword_retriever(articles):
    def retrieve(question, area, k=3):
        pool = [a for a in articles if a.get("area") == area] or articles
        bm25 = BM25Okapi([_tokenize(f"{a['title']} {a['text']}") for a in pool])
        scores = bm25.get_scores(_tokenize(question))
        ranked = sorted(range(len(pool)), key=lambda i: scores[i], reverse=True)[:k]
        return [{"id": pool[i]["id"], "title": pool[i]["title"], "text": pool[i]["text"]} for i in ranked]
    return retrieve

## 5. Nodes and edges

The prompts are in German because the answers are. Each node is a plain function, which makes it easy to test.

In [5]:
from langgraph.graph import StateGraph, START, END

ROUTER_PROMPT = (
    "Ordne den folgenden Fall genau einem Rechtsgebiet zu. Antworte nur mit einem Wort: "
    "'arbeitsrecht', 'mietrecht' oder 'ausserhalb'.\n\nFall: {question}"
)
SPECIALIST_PROMPT = (
    "Du bist Fachperson für Schweizer {area}. Beantworte den Fall ausschliesslich gestützt auf die Quellen.\n"
    "Gliederung: Normwahl, Subsumtion, Ergebnis, Quellen. Zitiere Artikel als 'Art. X OR'. "
    "Erfinde keine Artikel und keine Bundesgerichtsentscheide.\n\n"
    "Quellen:\n{sources}\n\nFall: {question}{feedback}"
)
REVIEWER_PROMPT = (
    "Prüfe den Entwurf streng. Stimmen Normwahl und Ergebnis mit den Quellen überein? "
    "Ist jeder zitierte Artikel in den Quellen enthalten? Fehlt etwas Wesentliches?\n"
    "Antworte in der ersten Zeile nur mit 'OK' oder mit 'REVISE: <Grund in einem Satz>'.\n\n"
    "Quellen:\n{sources}\n\nFall: {question}\n\nEntwurf:\n{draft}"
)

def format_sources(sources):
    return "\n".join(f"[{s['id']}] {s['title']}: {s['text']}" for s in sources)

def build_graph(llm, retrieve):
    def router(state):
        verdict = llm.invoke(ROUTER_PROMPT.format(question=state["question"])).content.lower()
        area = "mietrecht" if "miet" in verdict else "arbeitsrecht" if "arbeit" in verdict else "out_of_scope"
        return {"area": area, "revisions": 0}

    def specialist(state):
        sources = state.get("sources") or retrieve(state["question"], state["area"])
        feedback = f"\n\nKritik am letzten Entwurf: {state['review']}" if state.get("review") else ""
        draft = llm.invoke(SPECIALIST_PROMPT.format(
            area=state["area"], sources=format_sources(sources), question=state["question"], feedback=feedback,
        )).content
        return {"sources": sources, "draft": draft}

    def reviewer(state):
        verdict = llm.invoke(REVIEWER_PROMPT.format(
            sources=format_sources(state["sources"]), question=state["question"], draft=state["draft"],
        )).content.strip()
        first_line = verdict.splitlines()[0] if verdict else "REVISE: empty review"
        if first_line.upper().startswith("OK"):
            return {"review": "OK"}
        return {"review": first_line, "revisions": state.get("revisions", 0) + 1}

    def finalize(state):
        return {"answer": state["draft"], "needs_human": False}

    def human_review(state):
        note = "\n\n[An eine Fachperson übergeben: Der Prüfer war nach zwei Runden nicht zufrieden.]"
        return {"answer": state["draft"] + note, "needs_human": True}

    def out_of_scope(state):
        return {"answer": "Dieser Fall liegt ausserhalb von Arbeits- und Mietrecht. Dafür ist der Agent nicht gebaut.",
                "needs_human": False}

    def after_router(state):
        return "specialist" if state["area"] in ("arbeitsrecht", "mietrecht") else "out_of_scope"

    def after_review(state):
        if state["review"] == "OK":
            return "finalize"
        return "specialist" if state["revisions"] <= MAX_REVISIONS else "human_review"

    g = StateGraph(LegalState)
    for name, fn in [("router", router), ("specialist", specialist), ("reviewer", reviewer),
                     ("finalize", finalize), ("human_review", human_review), ("out_of_scope", out_of_scope)]:
        g.add_node(name, fn)
    g.add_edge(START, "router")
    g.add_conditional_edges("router", after_router, ["specialist", "out_of_scope"])
    g.add_edge("specialist", "reviewer")
    g.add_conditional_edges("reviewer", after_review, ["finalize", "specialist", "human_review"])
    for name in ["finalize", "human_review", "out_of_scope"]:
        g.add_edge(name, END)
    return g.compile()

## 6. Testing the logic without a model

A scripted fake model returns fixed answers in order. This checks the routing and the stop rule without any key
and without cost. Run it after every change to the graph.

In [7]:
from langchain_core.language_models.fake_chat_models import GenericFakeChatModel

def scripted(*replies):
    return GenericFakeChatModel(messages=iter(replies))

fake_sources = lambda question, area, k=3: [{"id": "OR-336c", "title": "Kündigung zur Unzeit", "text": "..."}]

# Case 1: reviewer rejects once, then accepts
run = build_graph(scripted("arbeitsrecht", "Entwurf 1", "REVISE: Sperrfrist fehlt", "Entwurf 2", "OK"),
                  fake_sources).invoke({"question": "Kündigung während Krankheit?"})
assert run["answer"] == "Entwurf 2" and run["revisions"] == 1 and not run["needs_human"]

# Case 2: reviewer never accepts, so after two revision rounds a human takes over
run = build_graph(scripted("arbeitsrecht", "E1", "REVISE: a", "E2", "REVISE: b", "E3", "REVISE: c"),
                  fake_sources).invoke({"question": "Schwieriger Fall"})
assert run["needs_human"] and run["revisions"] == 3

# Case 3: out of scope
run = build_graph(scripted("ausserhalb"), fake_sources).invoke({"question": "Erbrecht?"})
assert run["area"] == "out_of_scope"

print("All graph tests passed")

All graph tests passed


## 7. The graph as a diagram

In [9]:
graph_preview = build_graph(scripted("x"), fake_sources)
print(graph_preview.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	router(router)
	specialist(specialist)
	reviewer(reviewer)
	finalize(finalize)
	human_review(human_review)
	out_of_scope(out_of_scope)
	__end__([<p>__end__</p>]):::last
	__start__ --> router;
	reviewer -.-> finalize;
	reviewer -.-> human_review;
	reviewer -.-> specialist;
	router -.-> out_of_scope;
	router -.-> specialist;
	specialist --> reviewer;
	finalize --> __end__;
	human_review --> __end__;
	out_of_scope --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## 8. Running real cases

In [11]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL, api_key=os.environ["LITELLM_API_KEY"], temperature=0)
legal_agent = build_graph(llm, make_keyword_retriever(load_articles()))

for case in [
    "Ich arbeite seit drei Jahren in einer Firma und bin seit zwei Wochen krank. Mein Chef will mir kündigen. Darf er das?",
    "Mein Vermieter hat mir eine Mahnung geschickt, weil ich zwei Monatsmieten nicht bezahlt habe. Was droht mir?",
]:
    result = legal_agent.invoke({"question": case})
    print(f"Area: {result['area']} | revisions: {result.get('revisions')} | human: {result.get('needs_human')}")
    print(result["answer"], "\n" + "-" * 80)

Area: arbeitsrecht | revisions: 0 | human: False
**Normwahl**  
Die einschlägige Norm ist **Art. 336c OR** (Kündigungsschutz bei Krankheit).  

**Subsumtion**  

| Tatbestandsmerkmal | Gesetzliche Vorgabe (Art. 336c OR) | Tatsächliche Situation |
|--------------------|-----------------------------------|------------------------|
| **Dauer der Betriebszugehörigkeit** | Der Schutz gilt ab dem 2. Dienstjahr bis zum 5. Dienstjahr für **90 Tage** Krankheit ohne eigenes Verschulden. | Der Arbeitnehmer ist seit **drei Jahren** beschäftigt → fällt in den Schutzzeitraum von 90 Tagen. |
| **Dauer der Krankheit** | Während der Schutzfrist (max. 90 Tage im 2.–5. Dienstjahr) darf der Arbeitgeber **nicht kündigen**. | Der Arbeitnehmer ist seit **zwei Wochen** (14 Tage) krank – innerhalb der geschützten 90‑Tage‑Frist. |
| **Kündigung während der Sperrfrist** | Kündigungen, die **während** einer Sperrfrist erklärt werden, sind **nichtig** (Art. 336c Abs. 2). | Die Kündigung wird **während** der Sperrf

## Your turn

- Change `MAX_REVISIONS` to 0. What does the graph do now, and what does that mean for experiment F3?
- Replace `human_review` with LangGraph's `interrupt()` and a checkpointer, so a person can approve or edit the draft
  before it is sent (task T14).
- Swap the keyword retriever for `retrieve()` from notebook 02 or the MCP tools from notebook 03.

## Decision log

| Date | Decision | Rejected alternative | Reason |
|---|---|---|---|
| Oct 2026 | Router plus one specialist per area | One generalist agent | Smaller prompts per area, routing is measurable on its own. |
| Oct 2026 | Reviewer loop with at most 2 rounds, then a human | Unlimited loop | Cost and latency stay bounded. Unclear cases are not answered automatically. |
| | Human in the loop via `interrupt()`: | | |
| 13.10.2026 | Specialist answers as JSON (sachverhalt, rechtsfrage, normen, subsumtion, ergebnis, quellen), parsed with raw-text fallback | with_structured_output | Fake models in the offline tests and models without reliable tool calling (Apertus) cannot use it |

## Building block to take away

`LegalState` and `build_graph(llm, retrieve)` move into the repository as `agents/graph.py`. The tests in section 6
move into `tests/test_graph.py`.

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=9b9d17b7-b4fd-473b-beb2-f7ed180fbdd3' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>